# 🤖 Entrenamiento y Comparación de Modelos de Detección de Anomalías

Este notebook entrena y compara **Isolation Forest** y **Local Outlier Factor (LOF)** sobre el dataset `intel-cpu-dataset`, usando anomalías simuladas como verdad de referencia para evaluar cada modelo.

**Flujo del notebook:**
1. Instalación y carga
2. Preprocesamiento (imputación + escalado)
3. Simulación de anomalías
4. Entrenamiento de modelos
5. Comparación de métricas
6. Visualizaciones comparativas
7. Selección del modelo ganador
8. Exportación del modelo `.pkl`

---
**Universidad ECCI · Electiva II — DevOps · 2026**
**Autores:** Julian David Garzon Medina · Javier Stiven Amaya Devia


## 1. Instalación y carga

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib
print("✅ Dependencias listas")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
import joblib
import warnings
from datetime import datetime, timezone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

BLUE_DARK  = '#1F3864'
BLUE_MED   = '#2E75B6'
BLUE_LIGHT = '#9DC3E6'
ACCENT     = '#E74C3C'
GREEN      = '#27AE60'
ORANGE     = '#E67E22'

print("✅ Librerías importadas")


> Sube el archivo `intel_dataset.csv` cuando aparezca el botón.

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"✅ Dataset cargado: {df_raw.shape[0]:,} registros × {df_raw.shape[1]} columnas")
df_raw.head()


## 2. Preprocesamiento

Aplicamos la estrategia definida en el EDA: **imputación con mediana** para conservar
los 2.081 registros, seguida de **StandardScaler** para normalizar las features del modelo.


In [ ]:
NUM_COLS = [
    'cpu_usage', 'memory_usage', 'network_traffic',
    'power_consumption', 'num_executed_instructions',
    'execution_time', 'energy_efficiency'
]

MODEL_FEATURES = [
    'cpu_usage', 'memory_usage', 'network_traffic',
    'power_consumption', 'execution_time', 'energy_efficiency'
]

# Imputación con mediana
imputer = SimpleImputer(strategy='median')
df = df_raw.copy()
df[NUM_COLS] = imputer.fit_transform(df[NUM_COLS])

# Dataset de entrenamiento: solo las features del modelo
X_train = df[MODEL_FEATURES].copy()

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"Registros de entrenamiento: {len(X_train):,}")
print(f"Features:                   {MODEL_FEATURES}")
print(f"Nulos restantes:            {df[MODEL_FEATURES].isnull().sum().sum()}")
print(f"Shape escalado:             {X_train_scaled.shape}")


## 3. Simulación de Anomalías

Como el dataset no tiene etiquetas de anomalía, generamos **100 registros anómalos conocidos**
(25 por cada tipo de fallo) para usarlos como verdad de referencia en la evaluación.
Los modelos se entrenan **solo con los datos normales** y luego se evalúan sobre el conjunto completo.

| Tipo | Descripción | Representa |
|---|---|---|
| **CPU/Mem Saturación** | cpu≥92%, memory≥90% | Proceso runaway, fuga de memoria |
| **Network Spike** | network≥950, cpu≤8% | Exfiltración de datos, DDoS saliente |
| **Colapso Total** | Todas las métricas al límite | Fallo catastrófico del servidor |
| **Fantasma** | Todas las métricas en 0 | Servidor zombie, fallo silencioso |


In [ ]:
rng = np.random.default_rng(42)
N = 25  # anomalías por tipo

# Tipo 1: Saturación CPU + Memoria
anom_cpu_mem = pd.DataFrame({
    'cpu_usage':         rng.uniform(92, 100, N),
    'memory_usage':      rng.uniform(90, 100, N),
    'network_traffic':   rng.uniform(400, 650, N),
    'power_consumption': rng.uniform(380, 499, N),
    'execution_time':    rng.uniform(70,   99, N),
    'energy_efficiency': rng.uniform(0.8,   1, N),
})

# Tipo 2: Spike de red con CPU baja
anom_net_spike = pd.DataFrame({
    'cpu_usage':         rng.uniform(1,   8, N),
    'memory_usage':      rng.uniform(1,  12, N),
    'network_traffic':   rng.uniform(950,999, N),
    'power_consumption': rng.uniform(10,  40, N),
    'execution_time':    rng.uniform(0.5,  5, N),
    'energy_efficiency': rng.uniform(0.05,0.2,N),
})

# Tipo 3: Colapso total
anom_collapse = pd.DataFrame({
    'cpu_usage':         rng.uniform(97, 100, N),
    'memory_usage':      rng.uniform(97, 100, N),
    'network_traffic':   rng.uniform(950, 999, N),
    'power_consumption': rng.uniform(470, 499, N),
    'execution_time':    rng.uniform(92,   99, N),
    'energy_efficiency': rng.uniform(0.9,   1, N),
})

# Tipo 4: Servidor fantasma (valores casi cero)
anom_ghost = pd.DataFrame({
    'cpu_usage':         rng.uniform(0.01, 0.5,  N),
    'memory_usage':      rng.uniform(0.01, 0.5,  N),
    'network_traffic':   rng.uniform(0.1,  2,    N),
    'power_consumption': rng.uniform(0.1,  1,    N),
    'execution_time':    rng.uniform(0.0001,0.01,N),
    'energy_efficiency': rng.uniform(0.0001,0.01,N),
})

# Unir todo
anomalies   = pd.concat([anom_cpu_mem, anom_net_spike, anom_collapse, anom_ghost],
                         ignore_index=True)
anom_types  = (['CPU/Mem Saturación']*N + ['Network Spike']*N +
               ['Colapso Total']*N      + ['Fantasma']*N)

# Dataset completo con etiquetas de verdad
X_full       = pd.concat([X_train, anomalies], ignore_index=True)
X_full_scaled = scaler.transform(X_full)

# Etiquetas reales: 1 = normal, -1 = anomalía
y_true = np.array([1]*len(X_train) + [-1]*len(anomalies))

print(f"Registros normales:          {len(X_train):,}")
print(f"Anomalías simuladas:         {len(anomalies)} (25 por tipo × 4 tipos)")
print(f"Dataset completo:            {len(X_full):,}")
print(f"Proporción anomalías:        {len(anomalies)/len(X_full)*100:.1f}%")


## 4. Entrenamiento de Modelos

Entrenamos cada modelo **únicamente sobre los datos normales** (`X_train_scaled`).
La evaluación se realiza sobre el dataset completo que incluye las anomalías simuladas.


### 4.1 Isolation Forest

In [ ]:
# ── Isolation Forest ────────────────────────────────────────────────
t0 = time.time()
iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)
iso.fit(X_train_scaled)
t_if_train = time.time() - t0

# Predicción sobre el dataset completo
t0 = time.time()
pred_if  = iso.predict(X_full_scaled)
score_if = iso.decision_function(X_full_scaled)
t_if_inf = (time.time() - t0) / len(X_full) * 1000  # ms por registro

print(f"✅ Isolation Forest entrenado")
print(f"   Tiempo de entrenamiento:  {t_if_train:.3f}s")
print(f"   Tiempo inf. por registro: {t_if_inf:.4f}ms")
print(f"   Anomalías detectadas (total dataset): {(pred_if == -1).sum()}")


### 4.2 Local Outlier Factor (LOF)

In [ ]:
# ── Local Outlier Factor ────────────────────────────────────────────
# novelty=True permite predecir sobre datos nuevos (no vistos en entrenamiento)
t0 = time.time()
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    novelty=True,
    n_jobs=-1
)
lof.fit(X_train_scaled)
t_lof_train = time.time() - t0

t0 = time.time()
pred_lof  = lof.predict(X_full_scaled)
score_lof = lof.decision_function(X_full_scaled)
t_lof_inf = (time.time() - t0) / len(X_full) * 1000

print(f"✅ LOF entrenado")
print(f"   Tiempo de entrenamiento:  {t_lof_train:.3f}s")
print(f"   Tiempo inf. por registro: {t_lof_inf:.4f}ms")
print(f"   Anomalías detectadas (total dataset): {(pred_lof == -1).sum()}")


## 5. Comparación de Métricas

Calculamos las métricas de evaluación para cada modelo usando como verdad
las etiquetas que nosotros generamos (`y_true`).


In [ ]:
def calcular_metricas(pred, y_true, n_anomalias=100):
    """Calcula métricas de detección dado un vector de predicciones."""
    mask_anom   = y_true == -1
    mask_normal = y_true ==  1

    tp  = ((pred == -1) & mask_anom).sum()    # anomalías correctamente detectadas
    fp  = ((pred == -1) & mask_normal).sum()  # normales marcados como anomalía
    tn  = ((pred ==  1) & mask_normal).sum()  # normales correctamente identificados
    fn  = ((pred ==  1) & mask_anom).sum()    # anomalías no detectadas

    tasa_deteccion  = tp / n_anomalias * 100
    tasa_fp         = fp / mask_normal.sum() * 100
    precision       = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0

    return {
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Tasa detección (%)':    round(tasa_deteccion, 2),
        'Falsos positivos (%)':  round(tasa_fp, 2),
        'Precisión (%)':         round(precision, 2),
    }

metricas_if  = calcular_metricas(pred_if,  y_true)
metricas_lof = calcular_metricas(pred_lof, y_true)

# Tabla comparativa general
resumen = pd.DataFrame({
    'Métrica':              ['Tasa de detección (%)', 'Falsos positivos (%)',
                             'Precisión (%)', 'Tiempo entrenamiento (s)',
                             'Tiempo inferencia (ms/reg)', 'TP', 'FP', 'FN'],
    'Isolation Forest':     [metricas_if['Tasa detección (%)'],
                             metricas_if['Falsos positivos (%)'],
                             metricas_if['Precisión (%)'],
                             round(t_if_train, 3),
                             round(t_if_inf, 4),
                             metricas_if['TP'], metricas_if['FP'], metricas_if['FN']],
    'LOF':                  [metricas_lof['Tasa detección (%)'],
                             metricas_lof['Falsos positivos (%)'],
                             metricas_lof['Precisión (%)'],
                             round(t_lof_train, 3),
                             round(t_lof_inf, 4),
                             metricas_lof['TP'], metricas_lof['FP'], metricas_lof['FN']],
})

print("COMPARACIÓN GENERAL DE MODELOS:")
display(resumen)


In [ ]:
# Detección por tipo de anomalía
tipos   = ['CPU/Mem Saturación', 'Network Spike', 'Colapso Total', 'Fantasma']
n_base  = len(X_train)

resultados_tipo = []
for i, tipo in enumerate(tipos):
    start = n_base + i * 25
    end   = start + 25
    det_if  = (pred_if[start:end]  == -1).sum()
    det_lof = (pred_lof[start:end] == -1).sum()
    resultados_tipo.append({
        'Tipo de anomalía':    tipo,
        'IF detectadas':       f"{det_if}/25  ({det_if/25*100:.0f}%)",
        'LOF detectadas':      f"{det_lof}/25  ({det_lof/25*100:.0f}%)",
    })

df_tipo = pd.DataFrame(resultados_tipo)
print("DETECCIÓN POR TIPO DE ANOMALÍA:")
display(df_tipo)


## 6. Visualizaciones Comparativas

In [ ]:
# ── Gráfico 1: Métricas principales en barras ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparación de Modelos — Métricas Principales',
             fontsize=13, fontweight='bold')

metricas_nombres = ['Tasa detección (%)', 'Falsos positivos (%)', 'Precisión (%)']
valores_if  = [metricas_if[m]  for m in metricas_nombres]
valores_lof = [metricas_lof[m] for m in metricas_nombres]
titulos     = ['Tasa de Detección
(↑ mejor)', 'Falsos Positivos
(↓ mejor)', 'Precisión
(↑ mejor)']

for ax, titulo, v_if, v_lof in zip(axes, titulos, valores_if, valores_lof):
    bars = ax.bar(['Isolation Forest', 'LOF'], [v_if, v_lof],
                  color=[BLUE_MED, ORANGE], alpha=0.85, edgecolor='white', width=0.5)
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel('%')
    ax.set_ylim(0, 115)
    for bar, val in zip(bars, [v_if, v_lof]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1.5,
                f'{val}%', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 2: Detección por tipo de anomalía ───────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Tasa de Detección por Tipo de Anomalía',
             fontsize=13, fontweight='bold')

n_base  = len(X_train)
x       = np.arange(len(tipos))
width   = 0.35

det_if_pct  = [(pred_if[n_base+i*25:n_base+(i+1)*25]  == -1).sum()/25*100 for i in range(4)]
det_lof_pct = [(pred_lof[n_base+i*25:n_base+(i+1)*25] == -1).sum()/25*100 for i in range(4)]

bars1 = ax.bar(x - width/2, det_if_pct,  width, label='Isolation Forest',
               color=BLUE_MED, alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, det_lof_pct, width, label='LOF',
               color=ORANGE,   alpha=0.85, edgecolor='white')

for bar, val in list(zip(bars1, det_if_pct)) + list(zip(bars2, det_lof_pct)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1,
            f'{val:.0f}%', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(tipos, fontsize=10)
ax.set_ylabel('% Anomalías detectadas')
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.axhline(80, color=ACCENT, linestyle='--', linewidth=1.5, label='Umbral mínimo (80%)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 3: Distribución del anomaly score ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Distribución del Anomaly Score — Normal vs Anomalía',
             fontsize=13, fontweight='bold')

mask_n = y_true ==  1
mask_a = y_true == -1

for ax, scores, titulo, model_color in zip(
    axes,
    [score_if, score_lof],
    ['Isolation Forest', 'LOF'],
    [BLUE_MED, ORANGE]
):
    ax.hist(scores[mask_n], bins=50, alpha=0.6, color=model_color,
            label='Normal', density=True)
    ax.hist(scores[mask_a], bins=30, alpha=0.8, color=ACCENT,
            label='Anomalía simulada', density=True)
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Densidad')
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("💡 Mayor separación entre distribuciones → mejor capacidad discriminativa del modelo.")


In [ ]:
# ── Gráfico 4: Scatter CPU vs Memory coloreado por modelo ────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('CPU vs Memory — Anomalías detectadas por cada modelo',
             fontsize=13, fontweight='bold')

X_full_df = X_full.copy()

for ax, pred, titulo in zip(axes, [pred_if, pred_lof],
                             ['Isolation Forest', 'LOF']):
    # Normales correctos
    mask_tn = (pred ==  1) & (y_true ==  1)
    # Anomalías correctamente detectadas
    mask_tp = (pred == -1) & (y_true == -1)
    # Falsos positivos
    mask_fp = (pred == -1) & (y_true ==  1)
    # Anomalías no detectadas
    mask_fn = (pred ==  1) & (y_true == -1)

    ax.scatter(X_full_df['cpu_usage'][mask_tn], X_full_df['memory_usage'][mask_tn],
               c=BLUE_LIGHT, s=8,  alpha=0.3, label='Normal (TN)')
    ax.scatter(X_full_df['cpu_usage'][mask_tp], X_full_df['memory_usage'][mask_tp],
               c=ACCENT,     s=30, alpha=0.8, zorder=5, label=f'Anomalía detectada (TP={mask_tp.sum()})')
    ax.scatter(X_full_df['cpu_usage'][mask_fp], X_full_df['memory_usage'][mask_fp],
               c=ORANGE,     s=20, alpha=0.7, zorder=4, label=f'Falso positivo (FP={mask_fp.sum()})')
    ax.scatter(X_full_df['cpu_usage'][mask_fn], X_full_df['memory_usage'][mask_fn],
               c='purple',   s=30, alpha=0.8, zorder=5, marker='x',
               label=f'No detectada (FN={mask_fn.sum()})')

    ax.set_xlabel('CPU Usage (%)')
    ax.set_ylabel('Memory Usage (%)')
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=8, framealpha=0.8)

plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico 5: Radar / tabla visual de comparación final ─────────────
categorias  = ['Detección
(%)', 'Bajo FP
(100-FP%)', 'Precisión
(%)',
               'Vel. Train
(inv. norm.)', 'Vel. Inf.
(inv. norm.)']

# Normalizar velocidad: más rápido = mejor score (invertido y escalado a 100)
max_train = max(t_if_train, t_lof_train)
max_inf   = max(t_if_inf,   t_lof_inf)

scores_if = [
    metricas_if['Tasa detección (%)'],
    100 - metricas_if['Falsos positivos (%)'],
    metricas_if['Precisión (%)'],
    round((1 - t_if_train  / max_train) * 100, 1),
    round((1 - t_if_inf    / max_inf)   * 100, 1),
]
scores_lof = [
    metricas_lof['Tasa detección (%)'],
    100 - metricas_lof['Falsos positivos (%)'],
    metricas_lof['Precisión (%)'],
    round((1 - t_lof_train / max_train) * 100, 1),
    round((1 - t_lof_inf   / max_inf)   * 100, 1),
]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(categorias))
w = 0.3
bars1 = ax.bar(x - w/2, scores_if,  w, color=BLUE_MED, alpha=0.85, label='Isolation Forest')
bars2 = ax.bar(x + w/2, scores_lof, w, color=ORANGE,   alpha=0.85, label='LOF')

for bar, val in list(zip(bars1, scores_if)) + list(zip(bars2, scores_lof)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8,
            f'{val:.0f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(categorias, fontsize=10)
ax.set_ylabel('Score (100 = mejor)')
ax.set_ylim(0, 115)
ax.set_title('Comparación Multidimensional (mayor = mejor en todas las dimensiones)',
             fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(80, color=ACCENT, linestyle='--', linewidth=1, alpha=0.6)
plt.tight_layout()
plt.show()


## 7. Selección del Modelo Ganador

Con base en las métricas obtenidas, analizamos las fortalezas y debilidades de cada modelo
para seleccionar el más adecuado para integrar al pipeline del proyecto.


In [ ]:
# Tabla de decisión final
decision = pd.DataFrame({
    'Criterio':          ['Tasa de detección',
                          'Falsos positivos',
                          'Precisión',
                          'Velocidad de entrenamiento',
                          'Velocidad de inferencia',
                          'Detección CPU/Mem Sat.',
                          'Detección Net Spike',
                          'Detección Colapso Total',
                          'Detección Fantasma',
                          'Integración con pipeline',
                          'Interpretabilidad del score'],
    'Isolation Forest':  [f"{metricas_if['Tasa detección (%)']}%",
                          f"{metricas_if['Falsos positivos (%)']}%",
                          f"{metricas_if['Precisión (%)']}%",
                          f"{t_if_train:.3f}s",
                          f"{t_if_inf:.4f}ms",
                          f"{(pred_if[n_base:n_base+25]==-1).sum()}/25",
                          f"{(pred_if[n_base+25:n_base+50]==-1).sum()}/25",
                          f"{(pred_if[n_base+50:n_base+75]==-1).sum()}/25",
                          f"{(pred_if[n_base+75:n_base+100]==-1).sum()}/25",
                          '✅ Nativa scikit-learn',
                          '✅ Score continuo claro'],
    'LOF':               [f"{metricas_lof['Tasa detección (%)']}%",
                          f"{metricas_lof['Falsos positivos (%)']}%",
                          f"{metricas_lof['Precisión (%)']}%",
                          f"{t_lof_train:.3f}s",
                          f"{t_lof_inf:.4f}ms",
                          f"{(pred_lof[n_base:n_base+25]==-1).sum()}/25",
                          f"{(pred_lof[n_base+25:n_base+50]==-1).sum()}/25",
                          f"{(pred_lof[n_base+50:n_base+75]==-1).sum()}/25",
                          f"{(pred_lof[n_base+75:n_base+100]==-1).sum()}/25",
                          '✅ Nativa scikit-learn',
                          '⚠️ Score menos intuitivo'],
})

display(decision)

# Veredicto
print()
print("=" * 55)
print("VEREDICTO")
print("=" * 55)

dr_if_val  = metricas_if['Tasa detección (%)']
dr_lof_val = metricas_lof['Tasa detección (%)']
fp_if_val  = metricas_if['Falsos positivos (%)']
fp_lof_val = metricas_lof['Falsos positivos (%)']

if dr_lof_val > dr_if_val and fp_lof_val <= fp_if_val:
    ganador = "LOF"
    razon   = f"mayor tasa de detección ({dr_lof_val}% vs {dr_if_val}%) con falsos positivos similares o menores"
elif dr_if_val > dr_lof_val and fp_if_val <= fp_lof_val:
    ganador = "Isolation Forest"
    razon   = f"mayor tasa de detección ({dr_if_val}% vs {dr_lof_val}%) con falsos positivos similares o menores"
else:
    # LOF tiene mejor detección pero IF es más rápido — contexto decide
    ganador = "LOF"
    razon   = f"tasa de detección superior ({dr_lof_val}% vs {dr_if_val}%), priorizando cobertura ante fallos críticos"

print(f"  Modelo seleccionado: {ganador}")
print(f"  Razón:               {razon}")
print()
print("  Ambos modelos superan el umbral mínimo de 80% de detección.")
print("  El modelo seleccionado se exportará para integración con la API FastAPI.")


## 8. Exportación del Modelo

Exportamos el modelo ganador junto con el scaler y los metadatos del experimento.
El archivo `.pkl` generado es el que consumirá la API FastAPI.


In [ ]:
import os
from datetime import datetime, timezone

# Seleccionar modelo ganador
modelo_ganador = lof if ganador == "LOF" else iso
nombre_modelo  = ganador.replace(" ", "_").lower()
version        = datetime.now(timezone.utc).strftime("v%Y%m%d_%H%M%S")
nombre_archivo = f"modelo_{nombre_modelo}_{version}.pkl"

# Empaquetar modelo + scaler juntos
artefacto = {
    "model":   modelo_ganador,
    "scaler":  scaler,
    "features":MODEL_FEATURES,
    "version": version,
}

joblib.dump(artefacto, nombre_archivo)
print(f"✅ Modelo exportado: {nombre_archivo}")

# Guardar también con nombre fijo para la API
joblib.dump(artefacto, "isolation_forest.pkl")
print(f"✅ Copia fija guardada: isolation_forest.pkl")

# Metadatos del experimento
metadata = {
    "version":          version,
    "trained_at":       datetime.now(timezone.utc).isoformat(),
    "model_selected":   ganador,
    "dataset": {
        "records_train":    len(X_train),
        "records_anomalies":len(anomalies),
        "features":         MODEL_FEATURES,
        "preprocessing":    "SimpleImputer(median) + StandardScaler",
    },
    "hyperparameters": (
        {"n_estimators": 200, "contamination": 0.05, "random_state": 42}
        if ganador == "Isolation Forest"
        else {"n_neighbors": 20, "contamination": 0.05, "novelty": True}
    ),
    "metrics": {
        "tasa_deteccion_pct":    metricas_lof['Tasa detección (%)'] if ganador == "LOF" else metricas_if['Tasa detección (%)'],
        "falsos_positivos_pct":  metricas_lof['Falsos positivos (%)'] if ganador == "LOF" else metricas_if['Falsos positivos (%)'],
        "precision_pct":         metricas_lof['Precisión (%)'] if ganador == "LOF" else metricas_if['Precisión (%)'],
        "train_time_s":          round(t_lof_train if ganador == "LOF" else t_if_train, 4),
        "inf_ms_per_record":     round(t_lof_inf   if ganador == "LOF" else t_if_inf,   4),
    },
    "anomaly_types_evaluated": tipos,
    "detection_by_type": {
        t: int((pred_lof[n_base+i*25:n_base+(i+1)*25]==-1).sum()
               if ganador=="LOF"
               else (pred_if[n_base+i*25:n_base+(i+1)*25]==-1).sum())
        for i, t in enumerate(tipos)
    }
}

with open("metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadatos guardados: metadata.json")
print()
print(json.dumps(metadata["metrics"], indent=2))


In [ ]:
# Descargar los archivos generados
from google.colab import files

print("Descargando archivos...")
files.download(nombre_archivo)
files.download("isolation_forest.pkl")
files.download("metadata.json")
print("✅ Descarga iniciada — guarda estos archivos en la carpeta models/ del proyecto")


## 9. Resumen Final

### Métricas obtenidas

| Modelo | Tasa detección | Falsos positivos | Precisión | Inf. (ms/reg) |
|---|---|---|---|---|
| **Isolation Forest** | 92% | 5.0% | 46.9% | 0.037ms |
| **LOF** | 99% | 4.1% | 53.5% | 0.056ms |

### Detección por tipo de anomalía

| Tipo | Isolation Forest | LOF |
|---|---|---|
| CPU/Mem Saturación | 68% | 96% |
| Network Spike | 100% | 100% |
| Colapso Total | 100% | 100% |
| Fantasma | 100% | 100% |

### Próximos pasos

1. **Copiar `isolation_forest.pkl` y `metadata.json`** a la carpeta `models/` del repositorio
2. **La API FastAPI** (`main.py`) ya está preparada para cargar este artefacto automáticamente
3. **Desplegar la API** con Docker: `docker compose up anomaly-api`
4. **Actualizar el workflow de n8n** con el nodo HTTP Request apuntando a `POST /predict`

> Los archivos descargados son compatibles directamente con la API FastAPI ya desarrollada.
> No se requieren cambios en el código de la API.
